In [1]:
pip install datasets


Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("LocalDoc/azerbaijani-ner-dataset")
print(ds)
print(ds["train"][0])

/Users/nazrinaliyeva/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/nazrinaliyeva/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['index', 'tokens', 'ner_tags'],
        num_rows: 99545
    })
})
{'index': '640b71a8-014e-424b-96e1-80c74c9317bb', 'tokens': "['Komitədən', 'bildirilib', 'ki', ',', 'sovet', 'dövründə', 'Azərbaycanda', 'cəmi', '17', 'məscid', 'fəaliyyət', 'göstərirdisə', ',', 'dövlət', 'müstəqilliyinin', 'bərpasından', 'sonra', 'ölkədə', '814', 'məscid', 'tikilib', '.']", 'ner_tags': '[3, 0, 0, 0, 0, 0, 14, 0, 17, 0, 0, 0, 0, 3, 0, 0, 0, 14, 17, 0, 0, 0]'}


In [3]:
import ast
from datasets import load_dataset, Dataset, DatasetDict

raw_ds = load_dataset("LocalDoc/azerbaijani-ner-dataset")

def safe_parse_list(x):
    if x is None:
        return None
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        x = x.strip()
        if not x:
            return None
        return ast.literal_eval(x)
    return None

def normalize_tokens(x):
    x = safe_parse_list(x)
    if x is None:
        return None
    return [str(t) for t in x]

def normalize_tags(x):
    x = safe_parse_list(x)
    if x is None:
        return None
    try:
        return [int(t) for t in x]
    except Exception:
        return None

def convert_split(split, split_name):
    rows = []
    skipped = 0

    for i, ex in enumerate(split):
        tokens = normalize_tokens(ex.get("tokens"))
        ner_tags = normalize_tags(ex.get("ner_tags"))

        if tokens is None or ner_tags is None:
            skipped += 1
            continue

        if len(tokens) != len(ner_tags):
            skipped += 1
            continue

        rows.append({
            "index": str(ex.get("index")),
            "tokens": tokens,
            "ner_tags": ner_tags,
        })

    print(split_name, "kechilen/buraxilan setirler:", skipped)
    return Dataset.from_list(rows)

parts = {}
for split_name in raw_ds.keys():
    parts[split_name] = convert_split(raw_ds[split_name], split_name)

ds = DatasetDict(parts)

print(ds)
print(ds["train"][0])
print(type(ds["train"][0]["tokens"]))
print(type(ds["train"][0]["ner_tags"]))
print(type(ds["train"][0]["ner_tags"][0]))


train kechilen/buraxilan setirler: 3592
DatasetDict({
    train: Dataset({
        features: ['index', 'tokens', 'ner_tags'],
        num_rows: 95953
    })
})
{'index': '640b71a8-014e-424b-96e1-80c74c9317bb', 'tokens': ['Komitədən', 'bildirilib', 'ki', ',', 'sovet', 'dövründə', 'Azərbaycanda', 'cəmi', '17', 'məscid', 'fəaliyyət', 'göstərirdisə', ',', 'dövlət', 'müstəqilliyinin', 'bərpasından', 'sonra', 'ölkədə', '814', 'məscid', 'tikilib', '.'], 'ner_tags': [3, 0, 0, 0, 0, 0, 14, 0, 17, 0, 0, 0, 0, 3, 0, 0, 0, 14, 17, 0, 0, 0]}
<class 'list'>
<class 'list'>
<class 'int'>


In [4]:
split1 = ds["train"].train_test_split(test_size=0.1, seed=42)
split2 = split1["test"].train_test_split(test_size=0.5, seed=42)

final_ds = DatasetDict({
    "train": split1["train"],
    "validation": split2["train"],
    "test": split2["test"],
})

print(final_ds)


DatasetDict({
    train: Dataset({
        features: ['index', 'tokens', 'ner_tags'],
        num_rows: 86357
    })
    validation: Dataset({
        features: ['index', 'tokens', 'ner_tags'],
        num_rows: 4798
    })
    test: Dataset({
        features: ['index', 'tokens', 'ner_tags'],
        num_rows: 4798
    })
})


In [6]:
import json

id2label = {
    0: "O",
    1: "PERSON",
    2: "LOCATION",
    3: "ORGANISATION",
    4: "DATE",
    5: "TIME",
    6: "MONEY",
    7: "PERCENTAGE",
    8: "FACILITY",
    9: "PRODUCT",
    10: "EVENT",
    11: "ART",
    12: "LAW",
    13: "LANGUAGE",
    14: "GPE",
    15: "NORP",
    16: "ORDINAL",
    17: "CARDINAL",
    18: "DISEASE",
    19: "CONTACT",
    20: "ADAGE",
    21: "QUANTITY",
    22: "MISCELLANEOUS",
    23: "POSITION",
    24: "PROJECT",
}

def to_gliner_example(example):
    tokens = example["tokens"]
    tags = example["ner_tags"]

    ner = []
    current_label = None
    start_idx = None

    for i, tag_id in enumerate(tags):
        if tag_id == 0:
            if current_label is not None:
                ner.append([start_idx, i - 1, current_label])
                current_label = None
                start_idx = None
            continue

        label = id2label[tag_id].lower()

        if current_label is None:
            current_label = label
            start_idx = i
        elif label != current_label:
            ner.append([start_idx, i - 1, current_label])
            current_label = label
            start_idx = i

    if current_label is not None:
        ner.append([start_idx, len(tags) - 1, current_label])

    return {
        "tokenized_text": tokens,
        "ner": ner,
    }

def convert_split(split, drop_empty=True):
    rows = []
    for ex in split:
        row = to_gliner_example(ex)
        if drop_empty and len(row["ner"]) == 0:
            continue
        rows.append(row)
    return rows

train_data = convert_split(final_ds["train"], drop_empty=True)
val_data = convert_split(final_ds["validation"], drop_empty=True)
test_data = convert_split(final_ds["test"], drop_empty=False)

print(len(train_data), len(val_data), len(test_data))
print(train_data[0])

with open("train_gliner.json", "w", encoding="utf-8") as f:
    json.dump(train_data, f, ensure_ascii=False)

with open("val_gliner.json", "w", encoding="utf-8") as f:
    json.dump(val_data, f, ensure_ascii=False)

with open("test_gliner.json", "w", encoding="utf-8") as f:
    json.dump(test_data, f, ensure_ascii=False)


65491 3605 4798
{'tokenized_text': ['Heyvan', 'sahibinə', 'külli', 'miqdarda', 'maddi', 'ziyan', 'vurulub', '.'], 'ner': [[2, 3, 'quantity']]}


In [ ]:
pip install -U gliner accelerate datasets seqeval


Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 170 kB 692 kB/s eta 0:00:01
  Using cached accelerate-1.10.1-py3-none-any.whl (374 kB)
     |████████████████████████████████| 43 kB 440 kB/s eta 0:00:01
     |████████████████████████████████| 16.8 MB 2.0 MB/s eta 0:00:01
  Using cached torch-2.8.0-cp39-none-macosx_11_0_arm64.whl (73.6 MB)
     |████████████████████████████████| 11.1 MB 2.1 MB/s eta 0:00:01
     |████████████████████████████████| 309 kB 1.7 MB/s eta 0:00:01
     |████████████████████████████████| 30.3 MB 2.7 MB/s eta 0:00:011
  Using cached networkx-3.2.1-py3-none-any.whl (1.6 MB)
  Using cached sympy-1.14.0-py3-none-any.whl (6.3 MB)
  Using cached jinja2-3.1.6-py3-none-any.whl (134 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl (536 kB)
  Using cached markupsafe-3.0.3-cp39-cp39-macosx_11_0_arm64.whl (12 kB)
     |████████████████████████████████| 46 kB 4.2 MB/s eta 0:00:01
     |████████████████████

In [8]:
import gc
import torch
from gliner import GLiNER

gc.collect()
if torch.backends.mps.is_available():
    torch.mps.empty_cache()

MAX_TOKENS = 96

train_data_small = [
    x for x in train_data
    if 1 <= len(x["tokenized_text"]) <= MAX_TOKENS
][:10000]

val_data_small = [
    x for x in val_data
    if 1 <= len(x["tokenized_text"]) <= MAX_TOKENS
][:1000]

print(len(train_data_small), len(val_data_small))
print(max(len(x["tokenized_text"]) for x in train_data_small))


10000 1000
65


In [9]:
model = GLiNER.from_pretrained("urchade/gliner_multi-v2.1")

model.train_model(
    train_dataset=train_data_small,
    eval_dataset=val_data_small,
    output_dir="gliner_az_model",
    max_steps=300,
    learning_rate=2e-5,
    others_lr=1e-5,
    weight_decay=0.01,
    others_weight_decay=0.01,
    lr_scheduler_type="linear",
    warmup_ratio=0.1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    logging_steps=20,
    save_steps=150,
    save_total_limit=1,
)


/Users/nazrinaliyeva/Library/Python/3.9/lib/python/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 45990.18it/s]
/Users/nazrinaliyeva/Library/Python/3.9/lib/python/site-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
/Users/nazrinaliyeva/Library/Python/3.9/lib/python/site-packages/gliner/model.py:1145: FutureWarning: 

Step,Training Loss
20,7.508600


RuntimeError: MPS backend out of memory (MPS allocated: 5.31 GiB, other allocations: 14.32 GiB, max allowed: 20.13 GiB). Tried to allocate 732.73 MiB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).

## Datasetin genislendirilme prosesi



In [5]:
id2label = {
    0: "O",
    1: "PERSON",
    2: "LOCATION",
    3: "ORGANISATION",
    4: "DATE",
    5: "TIME",
    6: "MONEY",
    7: "PERCENTAGE",
    8: "FACILITY",
    9: "PRODUCT",
    10: "EVENT",
    11: "ART",
    12: "LAW",
    13: "LANGUAGE",
    14: "GPE",
    15: "NORP",
    16: "ORDINAL",
    17: "CARDINAL",
    18: "DISEASE",
    19: "CONTACT",
    20: "ADAGE",
    21: "QUANTITY",
    22: "MISCELLANEOUS",
    23: "POSITION",
    24: "PROJECT",
}


In [6]:
from collections import Counter

label_token_counts = Counter()
label_row_counts = Counter()
broken_rows = 0

for ex in final_ds["train"]:
    tokens = ex["tokens"]
    tags = ex["ner_tags"]

    if tokens is None or tags is None or len(tokens) != len(tags):
        broken_rows += 1
        continue

    labels_in_this_row = set()

    for tag in tags:
        if tag != 0:
            label = id2label[tag]
            label_token_counts[label] += 1
            labels_in_this_row.add(label)

    for label in labels_in_this_row:
        label_row_counts[label] += 1

print("broken row sayi:", broken_rows)
print("\nToken sayina gore label statistikasi:")
for label, count in label_token_counts.most_common():
    print(label, count)

print("\nRow sayina gore label statistikasi:")
for label, count in label_row_counts.most_common():
    print(label, count)


broken row sayi: 0

Token sayina gore label statistikasi:
ORGANISATION 56811
PERSON 48843
GPE 31590
MISCELLANEOUS 30217
DATE 23591
CARDINAL 19744
LOCATION 12196
POSITION 7727
PRODUCT 7201
EVENT 7153
FACILITY 7094
ORDINAL 5459
NORP 4414
ART 3915
MONEY 3490
QUANTITY 2852
DISEASE 2551
PROJECT 2119
TIME 1922
PERCENTAGE 1185
LANGUAGE 554
LAW 513
CONTACT 442
ADAGE 235

Row sayina gore label statistikasi:
ORGANISATION 26232
PERSON 22912
GPE 21395
DATE 15522
CARDINAL 14043
LOCATION 8766
POSITION 5939
EVENT 5596
MISCELLANEOUS 5225
ORDINAL 4876
FACILITY 4721
PRODUCT 4159
NORP 3770
ART 2391
MONEY 2245
QUANTITY 2078
DISEASE 2000
PROJECT 1646
TIME 1317
PERCENTAGE 892
LANGUAGE 433
LAW 358
CONTACT 313
ADAGE 193


In [7]:
import random

def show_example(ex):
    print("TEXT:", " ".join(ex["tokens"]))
    print("ENTITIES:")
    for token, tag in zip(ex["tokens"], ex["ner_tags"]):
        if tag != 0:
            print(f"  {token} -> {id2label[tag]}")
    print("-" * 50)

sample_ids = random.sample(range(len(final_ds["train"])), 10)

for idx in sample_ids:
    show_example(final_ds["train"][idx])


TEXT: Əgər şagird elliptik orbitlər haqqında kifayət qədər məlumatın təfərrüatlarını başqalarına izah edə bilirsə , deməli , yaxşı öyrənib .
ENTITIES:
--------------------------------------------------
TEXT: Allah bilir , siz bilmirsiniz !
ENTITIES:
--------------------------------------------------
TEXT: Müsəlman bir şəxs öz maddi vəziyyətinin yaxşı olub olmadığını özü müəyyən edir və qurbanı da buna əsasən kəsir ?
ENTITIES:
--------------------------------------------------
TEXT: “ Standartların tətbiq olunması üçün işləri başa çatdırmalıyıq ”
ENTITIES:
  Standartların -> MISCELLANEOUS
  tətbiq -> MISCELLANEOUS
  olunması -> MISCELLANEOUS
  işləri -> MISCELLANEOUS
--------------------------------------------------
TEXT: Azərbaycan Respublikasının Prezidenti İlham Əliyev və birinci xanım Mehriban Əliyeva iyunun 24-də 1 saylı Gəncə Şəhər Xəstəxanasının modul tipli infeksion xəstəliklər korpusunun açılışında iştirak ediblər .
ENTITIES:
  Azərbaycan -> GPE
  Respublikasının -> ORGANISATI